In [1]:
import json
import pandas as pd
from tqdm import tqdm
import re
import numpy as np
import matplotlib.pyplot as plt


## Drugs @ FDA

In [40]:
fda_drugs_file = "/shares/animalwelfare.crs.uzh/fda_files/2025_11_11/drug-drugsfda-0001-of-0001.json"

In [41]:

# Load the FDA JSON file
with open(fda_drugs_file, "r") as f:
    data = json.load(f)

# Initialize list for filtered results
orig_records = []

# Loop over results
for result in data.get("results", []):
    submissions = result.get("submissions", [])
    
    # Find the ORIG submission(s)
    orig_subs = [s for s in submissions if s.get("submission_type") == "ORIG"]
    if not orig_subs:
        continue

    orig = orig_subs[0]
    submission_date = orig.get("submission_status_date")
    year = submission_date[:4] if submission_date else None
    # Extract base info
    record = {
        "application_number": result.get("application_number"),
        "sponsor_name": result.get("sponsor_name"),
        "submission_type": orig.get("submission_type"),
        "submission_status": orig.get("submission_status"),
        "submission_status_date": submission_date,
        "submission_class_code_description": orig.get("submission_class_code_description"),
        "year": year
    }

    # Extract first product if exists
    products = result.get("products", [])
    if products:
        p = products[0]
        record.update({
            "brand_name": p.get("brand_name"),
            "dosage_form": p.get("dosage_form"),
            "route": p.get("route"),
            "marketing_status": p.get("marketing_status"),
            "active_ingredients": ", ".join(
                f"{a.get('name')}" for a in p.get("active_ingredients", [])
            )
        })

    # If openfda section exists, flatten a few key fields
    openfda = result.get("openfda", {})
    if openfda:
        record.update({
            "openfda_brand_name": ", ".join(openfda.get("brand_name", [])),
            "openfda_generic_name": ", ".join(openfda.get("generic_name", [])),
            "openfda_route": ", ".join(openfda.get("route", [])),
            "openfda_substance_name": ", ".join(openfda.get("substance_name", []))
        })

    orig_records.append(record)

# Convert to DataFrame for easier viewing/analysis
df_orig = pd.DataFrame(orig_records)

In [42]:
df_orig.shape

(25712, 16)

In [43]:
df_orig.submission_status.value_counts()

submission_status
AP    24636
TA     1075
Name: count, dtype: int64

In [44]:
# Show summary
print("Found", len(df_orig), "ORIG applications")
df_orig.head()

Found 25712 ORIG applications


,application_number,sponsor_name,submission_type,submission_status,submission_status_date,submission_class_code_description,year,brand_name,dosage_form,route,marketing_status,active_ingredients,openfda_brand_name,openfda_generic_name,openfda_route,openfda_substance_name
0,ANDA076194,WATSON LABS,ORIG,AP,20020701,None,2002,LISINOPRIL AND HYDROCHLOROTHIAZIDE,TABLET,ORAL,Prescription,"HYDROCHLOROTHIAZIDE, LISINOPRIL",LISINOPRIL AND HYDROCHLOROTHIAZIDE,LISINOPRIL AND HYDROCHLOROTHIAZIDE,ORAL,"HYDROCHLOROTHIAZIDE, LISINOPRIL"
1,ANDA076206,ROCKWELL MEDCL,ORIG,AP,20030917,None,2003,CALCITRIOL,INJECTABLE,INJECTION,Discontinued,CALCITRIOL,NaN,NaN,NaN,NaN
2,ANDA076212,APOTEX,ORIG,AP,20040616,None,2004,CARBIDOPA AND LEVODOPA,"TABLET, EXTENDED RELEASE",ORAL,Discontinued,"CARBIDOPA, LEVODOPA",NaN,NaN,NaN,NaN
3,ANDA076215,FOUGERA PHARMS,ORIG,AP,20031209,None,2003,BETAMETHASONE DIPROPIONATE,"CREAM, AUGMENTED",TOPICAL,Prescription,BETAMETHASONE DIPROPIONATE,NaN,NaN,NaN,NaN
4,ANDA076224,PHARMOBEDIENT,ORIG,AP,20030509,None,2003,FLUTAMIDE,CAPSULE,ORAL,Discontinued,FLUTAMIDE,NaN,NaN,NaN,NaN


In [45]:
df_orig[df_orig['active_ingredients']=="CLADRIBINE"]

,application_number,sponsor_name,submission_type,submission_status,submission_status_date,submission_class_code_description,year,brand_name,dosage_form,route,marketing_status,active_ingredients,openfda_brand_name,openfda_generic_name,openfda_route,openfda_substance_name
2062,NDA022561,EMD SERONO INC,ORIG,AP,20190329,Type 3 - New Dosage Form,2019,MAVENCLAD,TABLET,ORAL,Prescription,CLADRIBINE,MAVENCLAD,CLADRIBINE,ORAL,CLADRIBINE
8585,ANDA200510,PHARMOBEDIENT,ORIG,AP,20111006,Not Applicable,2011,CLADRIBINE,INJECTABLE,INJECTION,Discontinued,CLADRIBINE,NaN,NaN,NaN,NaN
11798,NDA020229,JANSSEN PHARMS,ORIG,AP,19930226,Type 1 - New Molecular Entity,1993,LEUSTATIN,INJECTABLE,INJECTION,Discontinued,CLADRIBINE,NaN,NaN,NaN,NaN
12849,ANDA075405,HIKMA,ORIG,AP,20000228,None,2000,CLADRIBINE,INJECTABLE,INJECTION,Prescription,CLADRIBINE,CLADRIBINE,CLADRIBINE,INTRAVENOUS,CLADRIBINE
20606,ANDA076571,FRESENIUS KABI USA,ORIG,AP,20040422,None,2004,CLADRIBINE,INJECTABLE,INJECTION,Prescription,CLADRIBINE,CLADRIBINE,CLADRIBINE,INTRAVENOUS,CLADRIBINE
24464,ANDA210856,HISUN PHARM HANGZHOU,ORIG,AP,20191125,None,2019,CLADRIBINE,INJECTABLE,INJECTION,Prescription,CLADRIBINE,CLADRIBINE,CLADRIBINE,INTRAVENOUS,CLADRIBINE


## Label

In [46]:
file_nr = "01"
drug_label_file = f"/shares/animalwelfare.crs.uzh/fda_files/2025_11_11/drug-label-00{file_nr}-of-0013.json"

# Load your drug label JSON (replace filename with yours)
with open(drug_label_file, "r") as f:
    data = json.load(f)

results = data.get("results", [])

records = []

for result in tqdm(results, desc="Processing FDA drug labels", unit="label"):
    openfda = result.get("openfda", {})
    application_numbers = openfda.get("application_number", [])

    # ✅ Skip if no application_number
    if not application_numbers:
        continue

    clinical_text = " ".join(result.get("clinical_studies", []))
    nct_ids = re.findall(r"NCT\s*[-:]?\s*\d{6,8}", clinical_text, flags=re.IGNORECASE)

    # --- Extract only the first sentence from indications_and_usage
    indications_text = " ".join(result.get("indications_and_usage", []))
    match = re.match(r"^(.*?)(?<!\d)\.\s", indications_text)
    first_sentence = match.group(1).strip() if match else indications_text.strip()

    record = {
        "application_number": ", ".join(openfda.get("application_number", [])),
        "label_brand_name": ", ".join(openfda.get("brand_name", [])),
        "label_generic_name": ", ".join(openfda.get("generic_name", [])),
        "label_manufacturer_name": ", ".join(openfda.get("manufacturer_name", [])),
        "label_substance_name": ", ".join(openfda.get("substance_name", [])),
        "indications_first_sent": first_sentence,
        "indications_and_usage": indications_text,
        "clinical_studies": nct_ids,
    }

    records.append(record)

# Convert to DataFrame
df_labels = pd.DataFrame(records)
df_orig = df_orig.merge(df_labels, on="application_number", how="left")

Processing FDA drug labels: 100%|██████████| 20000/20000 [00:00<00:00, 151991.49label/s]


In [47]:
df_labels

,application_number,label_brand_name,label_generic_name,label_manufacturer_name,label_substance_name,indications_first_sent,indications_and_usage,clinical_studies
0,ANDA209057,Vardenafil Hydrochloride,VARDENAFIL HYDROCHLORIDE,Bryant Ranch Prepack,VARDENAFIL HYDROCHLORIDE,1 INDICATIONS AND USAGE Vardenafil hydrochlori...,1 INDICATIONS AND USAGE Vardenafil hydrochlori...,[]
1,ANDA211610,ACETAMINOPHEN AND CODEINE PHOSPHATE,ACETAMINOPHEN AND CODEINE PHOSPHATE,Eywa Pharma Inc,"ACETAMINOPHEN, CODEINE PHOSPHATE",INDICATIONS AND USAGE Acetaminophen and codein...,INDICATIONS AND USAGE Acetaminophen and codein...,[]
2,ANDA214403,"Dextroamphetamine Saccharate, Amphetamine Aspa...","DEXTROAMPHETAMINE SULFATE, DEXTROAMPHETAMINE S...","Golden State Medical Supply, Inc.","AMPHETAMINE ASPARTATE MONOHYDRATE, AMPHETAMINE...",1 INDICATIONS AND USAGE Dextroamphetamine sacc...,1 INDICATIONS AND USAGE Dextroamphetamine sacc...,[]
3,ANDA211355,OMEGA-3-ACID ETHYL ESTERS,OMEGA-3-ACID ETHYL ESTERS,Procaps S A,OMEGA-3-ACID ETHYL ESTERS,1 INDICATION S AND USAGE Omega-3-Acid Ethyl Es...,1 INDICATION S AND USAGE Omega-3-Acid Ethyl Es...,[]
4,ANDA076674,LISINOPRIL and HYDROCHLOROTHIAZIDE,LISINOPRIL AND HYDROCHLOROTHIAZIDE,"GSMS, Incorporated","HYDROCHLOROTHIAZIDE, LISINOPRIL",INDICATIONS AND USAGE Lisinopril and Hydrochlo...,INDICATIONS AND USAGE Lisinopril and Hydrochlo...,[]
...,...,...,...,...,...,...,...,...
4480,M030,Salicylic Acid,LIQUID CORN AND CALLUS REMOVER,Chain Drug Marketing Association Inc.,SALICYLIC ACID,​Uses for the removal of corns and calluses re...,​Uses for the removal of corns and calluses re...,[]
4481,ANDA075614,terazosin,TERAZOSIN,Heritage Pharmaceuticals Inc. d/b/a Avet Pharm...,TERAZOSIN HYDROCHLORIDE,INDICATIONS AND USAGE Terazosin capsules are i...,INDICATIONS AND USAGE Terazosin capsules are i...,[]
4482,505G(a)(3),GOJO Antibacterial Plum Foam Handwash,BENZALKONIUM CHLORIDE,"GOJO Industries, Inc.",BENZALKONIUM CHLORIDE,Uses • Handwash to help decrease bacteria on t...,Uses • Handwash to help decrease bacteria on t...,[]
4483,505G(a)(3),PURELL HEALTHY SP 0.5pct BAK Antimicrobial Foam,BENZALKONIUM CHLORIDE,"GOJO Industries, Inc.",BENZALKONIUM CHLORIDE,Uses • Handwash to help decrease bacteria on t...,Uses • Handwash to help decrease bacteria on t...,[]


In [48]:
import json
import re
from pathlib import Path

import pandas as pd
from tqdm import tqdm

# --- only these columns will be written into df_orig
label_cols = [
    "label_brand_name",
    "label_generic_name",
    "label_manufacturer_name",
    "label_substance_name",
    "indications_first_sent",
    "indications_and_usage",
    "clinical_studies",
]

# ensure df_orig has target columns (so .update works cleanly)
for col in label_cols:
    if col not in df_orig.columns:
        df_orig[col] = pd.NA

# regex: first sentence without splitting on numbered list markers like "1."
FIRST_SENTENCE_RE = re.compile(r"^(.*?)(?<!\d)\.\s")

def first_sentence(text: str) -> str:
    if not text:
        return ""
    m = FIRST_SENTENCE_RE.match(text)
    return (m.group(1) if m else text).strip()

# --- iterate files 01..13, process and merge immediately
base_dir = Path("/shares/animalwelfare.crs.uzh/fda_files/2025_11_11")

for i in tqdm(range(1, 14), desc="Processing label files", unit="file"):
    file_nr = f"{i:02d}"
    drug_label_file = base_dir / f"drug-label-00{file_nr}-of-0013.json"

    with open(drug_label_file, "r") as f:
        data = json.load(f)

    results = data.get("results", [])

    # build minimal records for THIS file only
    records = []
    for result in tqdm(results, desc=f"Parsing {drug_label_file.name}", unit="label", leave=False):
        openfda = result.get("openfda", {}) or {}
        application_numbers = openfda.get("application_number", []) or []

        # Skip if no application_number
        if not application_numbers:
            continue

        # clinical trials: tolerant NCT regex (as you wrote)
        clinical_text = " ".join(result.get("clinical_studies", []) or [])
        nct_ids = re.findall(r"NCT\s*[-:]?\s*\d{6,8}", clinical_text, flags=re.IGNORECASE)

        # indications: first sentence only (don’t split on "1.")
        indications_text = " ".join(result.get("indications_and_usage", []) or [])
        match = re.match(r"^(.*?)(?<!\d)\.\s", indications_text)
        first_sent = match.group(1).strip() if match else indications_text.strip()

        # one row per application_number (clean merge key)
        for app_no in application_numbers:
            records.append({
                "application_number": app_no,
                "label_brand_name": ", ".join(openfda.get("brand_name", []) or []),
                "label_generic_name": ", ".join(openfda.get("generic_name", []) or []),
                "label_manufacturer_name": ", ".join(openfda.get("manufacturer_name", []) or []),
                "label_substance_name": ", ".join(openfda.get("substance_name", []) or []),
                "indications_first_sent": first_sent,
                "indications_and_usage": indications_text,
                "clinical_studies": nct_ids,  # keep as list
            })

    # turn this file's records into a df, dedupe by application_number to avoid row explosion
    df_labels_file = pd.DataFrame(records)
    if not df_labels_file.empty:
        df_labels_file = df_labels_file.drop_duplicates(subset=["application_number"], keep="first")
        # in-place update (no _x/_y columns, no row duplication)
        df_orig = df_orig.set_index("application_number")
        df_orig.update(df_labels_file.set_index("application_number")[label_cols])
        df_orig = df_orig.reset_index()

print("✅ Finished merging each label file into df_orig (on the fly).")


Processing label files: 100%|██████████| 13/13 [00:51<00:00,  4.00s/file]                       

✅ Finished merging each label file into df_orig (on the fly).


In [49]:
df_orig

,application_number,sponsor_name,submission_type,submission_status,submission_status_date,submission_class_code_description,year,brand_name,dosage_form,route,...,openfda_generic_name,openfda_route,openfda_substance_name,label_brand_name,label_generic_name,label_manufacturer_name,label_substance_name,indications_first_sent,indications_and_usage,clinical_studies
0,ANDA076194,WATSON LABS,ORIG,AP,20020701,None,2002,LISINOPRIL AND HYDROCHLOROTHIAZIDE,TABLET,ORAL,...,LISINOPRIL AND HYDROCHLOROTHIAZIDE,ORAL,"HYDROCHLOROTHIAZIDE, LISINOPRIL",Lisinopril and Hydrochlorothiazide,LISINOPRIL AND HYDROCHLOROTHIAZIDE,"Actavis Pharma, Inc.","HYDROCHLOROTHIAZIDE, LISINOPRIL",INDICATIONS AND USAGE Lisinopril and Hydrochlo...,INDICATIONS AND USAGE Lisinopril and Hydrochlo...,[]
1,ANDA076206,ROCKWELL MEDCL,ORIG,AP,20030917,None,2003,CALCITRIOL,INJECTABLE,INJECTION,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ANDA076212,APOTEX,ORIG,AP,20040616,None,2004,CARBIDOPA AND LEVODOPA,"TABLET, EXTENDED RELEASE",ORAL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ANDA076215,FOUGERA PHARMS,ORIG,AP,20031209,None,2003,BETAMETHASONE DIPROPIONATE,"CREAM, AUGMENTED",TOPICAL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ANDA076224,PHARMOBEDIENT,ORIG,AP,20030509,None,2003,FLUTAMIDE,CAPSULE,ORAL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26301,ANDA076154,MYLAN,ORIG,AP,20030820,None,2003,LORATADINE,TABLET,ORAL,...,LORATADINE,ORAL,LORATADINE,Loratadine,LORATADINE,Mylan Pharmaceuticals Inc.,LORATADINE,Uses temporarily relieves these symptoms due t...,Uses temporarily relieves these symptoms due t...,[]
26302,ANDA076155,APOTEX INC,ORIG,AP,20030418,None,2003,IPRATROPIUM BROMIDE,"SPRAY, METERED",NASAL,...,IPRATROPIUM BROMIDE,NASAL,IPRATROPIUM BROMIDE,Ipratropium Bromide,IPRATROPIUM BROMIDE,Apotex Corp.,IPRATROPIUM BROMIDE,INDICATIONS AND USAGE Ipratropium bromide nasa...,INDICATIONS AND USAGE Ipratropium bromide nasa...,[]
26303,ANDA076162,WATSON LABS TEVA,ORIG,AP,20041014,None,2004,CARBOPLATIN,INJECTABLE,INJECTION,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
26304,ANDA076163,DR REDDYS,ORIG,AP,20030905,None,2003,AMIODARONE HYDROCHLORIDE,INJECTABLE,INJECTION,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [50]:
df_orig[df_orig['active_ingredients']=="CLADRIBINE"]

,application_number,sponsor_name,submission_type,submission_status,submission_status_date,submission_class_code_description,year,brand_name,dosage_form,route,...,openfda_generic_name,openfda_route,openfda_substance_name,label_brand_name,label_generic_name,label_manufacturer_name,label_substance_name,indications_first_sent,indications_and_usage,clinical_studies
2116,NDA022561,EMD SERONO INC,ORIG,AP,20190329,Type 3 - New Dosage Form,2019,MAVENCLAD,TABLET,ORAL,...,CLADRIBINE,ORAL,CLADRIBINE,Mavenclad,CLADRIBINE,"EMD Serono, Inc.",CLADRIBINE,1 INDICATIONS AND USAGE MAVENCLAD is indicated...,1 INDICATIONS AND USAGE MAVENCLAD is indicated...,[NCT00213135]
8756,ANDA200510,PHARMOBEDIENT,ORIG,AP,20111006,Not Applicable,2011,CLADRIBINE,INJECTABLE,INJECTION,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12041,NDA020229,JANSSEN PHARMS,ORIG,AP,19930226,Type 1 - New Molecular Entity,1993,LEUSTATIN,INJECTABLE,INJECTION,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13100,ANDA075405,HIKMA,ORIG,AP,20000228,None,2000,CLADRIBINE,INJECTABLE,INJECTION,...,CLADRIBINE,INTRAVENOUS,CLADRIBINE,Cladribine,CLADRIBINE,Hikma Pharmaceuticals USA Inc.,CLADRIBINE,"INDICATIONS AND USAGE Cladribine Injection, US...","INDICATIONS AND USAGE Cladribine Injection, US...",[]
21084,ANDA076571,FRESENIUS KABI USA,ORIG,AP,20040422,None,2004,CLADRIBINE,INJECTABLE,INJECTION,...,CLADRIBINE,INTRAVENOUS,CLADRIBINE,Cladribine,CLADRIBINE,"Fresenius Kabi USA, LLC",CLADRIBINE,"INDICATIONS AND USAGE: Cladribine Injection, U...","INDICATIONS AND USAGE: Cladribine Injection, U...",[]
25046,ANDA210856,HISUN PHARM HANGZHOU,ORIG,AP,20191125,None,2019,CLADRIBINE,INJECTABLE,INJECTION,...,CLADRIBINE,INTRAVENOUS,CLADRIBINE,Cladribine,CLADRIBINE,"Hisun Pharmaceuticals USA, Inc.",CLADRIBINE,"INDICATIONS FOR USE Cladribine Injection, USP ...","INDICATIONS FOR USE Cladribine Injection, USP ...",[]


In [51]:
df_orig.to_csv("out/FDA_drugs_labels_full.csv",index=False)
df_orig.to_csv("/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/10_use_case_disease_focus/out/FDA_drugs_labels_full.csv",index=False)

## NDAs vs ANDAs

In [91]:
df_orig_ndas = df_orig[df_orig['application_number'].str.startswith("NDA")][['application_number', 'active_ingredients', 'brand_name', 'year', 'sponsor_name', 'submission_class_code_description', 'submission_status','indications_first_sent']]
df_orig_ndas = df_orig_ndas[df_orig_ndas['submission_status']=='AP']
df_orig_ndas.shape

(5325, 8)

In [92]:
df_orig_ndas["active_ingredients_split"] = (
    df_orig_ndas["active_ingredients"]
    .astype(str)
    .str.split(r",\s*|\|\|")  # Matches comma (+ optional space) OR literal ||
)
df_orig_ndas = df_orig_ndas.explode("active_ingredients_split", ignore_index=True)
df_orig_ndas.shape

(6808, 9)

In [93]:
df_orig_ndas.head()

,application_number,active_ingredients,brand_name,year,sponsor_name,submission_class_code_description,submission_status,indications_first_sent,active_ingredients_split
0,NDA017446,HEXACHLOROPHENE,PHISO-SCRUB,1977,SANOFI AVENTIS US,Type 5 - New Formulation or New Manufacturer,AP,NaN,HEXACHLOROPHENE
1,NDA017453,DIAZOXIDE,PROGLYCEM,1976,TEVA BRANDED PHARM,Type 3 - New Dosage Form,AP,INDICATIONS AND USAGE PROGLYCEM is indicated f...,DIAZOXIDE
2,NDA017476,POTASSIUM CHLORIDE,SLOW-K,1975,NOVARTIS,Type 5 - New Formulation or New Manufacturer,AP,NaN,POTASSIUM CHLORIDE
3,NDA017481,MEBENDAZOLE,VERMOX,1974,JANSSEN PHARMS,Type 1 - New Molecular Entity,AP,NaN,MEBENDAZOLE
4,NDA017513,STERILE WATER FOR IRRIGATION,STERILE WATER IN PLASTIC CONTAINER,1974,OTSUKA ICU MEDCL,Type 5 - New Formulation or New Manufacturer,AP,INDICATIONS AND USAGE Sterile Water for Irriga...,STERILE WATER FOR IRRIGATION


In [94]:
df_orig_ndas.active_ingredients.nunique(), df_orig_ndas.active_ingredients_split.nunique()

(2500, 2219)

In [95]:
df_orig_andas = df_orig[df_orig['application_number'].str.startswith("ANDA")][['application_number', 'active_ingredients', 'brand_name', 'year', 'sponsor_name', 'submission_class_code_description', 'submission_status','indications_first_sent']]
df_orig_andas = df_orig_andas[df_orig_andas['submission_status']=='AP']

df_orig_andas.shape

(19450, 8)

In [96]:
df_orig_andas["active_ingredients_split"] = (
    df_orig_andas["active_ingredients"]
    .astype(str)
    .str.split(r",\s*|\|\|")  # Matches comma (+ optional space) OR literal ||
)
df_orig_andas = df_orig_andas.explode("active_ingredients_split", ignore_index=True)
df_orig_andas.shape

(22163, 9)

In [97]:
df_orig_andas.active_ingredients.nunique(), df_orig_andas.active_ingredients_split.nunique()

(1418, 1258)

In [98]:
df_orig_ndas[df_orig_ndas['active_ingredients_split']=="TIGECYCLINE"]

,application_number,active_ingredients,brand_name,year,sponsor_name,submission_class_code_description,submission_status,indications_first_sent,active_ingredients_split
921,NDA208744,TIGECYCLINE,TIGECYCLINE,2018,ACCORD HLTHCARE INC,Type 5 - New Formulation or New Manufacturer,AP,NaN,TIGECYCLINE
2336,NDA211158,TIGECYCLINE,TIGECYCLINE,2018,AMNEAL,Type 5 - New Formulation or New Manufacturer,AP,1 INDICATIONS AND USAGE Tigecycline is a tetra...,TIGECYCLINE
5222,NDA021821,TIGECYCLINE,TYGACIL,2005,PF PRISM CV,Type 1 - New Molecular Entity,AP,1 INDICATIONS AND USAGE TYGACIL is a tetracycl...,TIGECYCLINE
5581,NDA205645,TIGECYCLINE,TIGECYCLINE,2016,FRESENIUS KABI USA,Type 5 - New Formulation or New Manufacturer,AP,1 INDICATIONS AND USAGE Tigecycline for inject...,TIGECYCLINE


In [99]:
set1 = set(df_orig_andas['active_ingredients_split'])
set2 = set(df_orig_ndas['active_ingredients_split'])

# Items in set1 but not in set2
difference = set1 - set2
len(difference)

60

In [108]:
# 1. Create sets using tuples (tuples are hashable, lists are not)
set1 = set(df_orig_andas['active_ingredients_split'].map(tuple))
set2 = set(df_orig_ndas['active_ingredients_split'].map(tuple))

# 2. Find the difference
difference = set1 - set2

# 3. Filter the original DataFrame
# We check if the tuple version of the row is in our "difference" set
df_only_in_andas = df_orig_andas[
    df_orig_andas['active_ingredients_split'].map(tuple).isin(difference)
]
df_only_in_andas

,application_number,active_ingredients,brand_name,year,sponsor_name,submission_class_code_description,submission_status,indications_first_sent,active_ingredients_split
204,ANDA078293,Oxybutynin Chloride,Oxybutynin Chloride,2007,MYLAN PHARMS INC,None,AP,NaN,Oxybutynin Chloride
294,ANDA080188,TESTOSTERONE PROPIONATE,TESTOSTERONE PROPIONATE,1974,WATSON LABS,None,AP,NaN,TESTOSTERONE PROPIONATE
306,ANDA080767,METHYLTESTOSTERONE,METHYLTESTOSTERONE,1974,IMPAX LABS,None,AP,INDICATIONS AND USAGE 1. Males Androgens are i...,METHYLTESTOSTERONE
330,ANDA083213,"HYDROCORTISONE ACETATE, PRAMOXINE HYDROCHLORIDE",PRAMOSONE,1973,FERNDALE LABS,None,AP,NaN,PRAMOXINE HYDROCHLORIDE
332,ANDA083247,PENTOBARBITAL SODIUM,NEMBUTAL,1982,SCIEGEN PHARMS INC,None,AP,NaN,PENTOBARBITAL SODIUM
...,...,...,...,...,...,...,...,...,...
21704,ANDA061655,KANAMYCIN SULFATE,KANTREX,1973,APOTHECON,None,AP,NaN,KANAMYCIN SULFATE
21733,ANDA062683,CEPHRADINE,CEPHRADINE,1987,TEVA,None,AP,NaN,CEPHRADINE
21735,ANDA062693,CEPHRADINE,CEPHRADINE,1987,TEVA,None,AP,NaN,CEPHRADINE
21739,ANDA062859,CEPHRADINE,CEPHRADINE,1988,BARR,None,AP,NaN,CEPHRADINE


In [100]:
overlaps = set1 & set2
len(overlaps), list(overlaps)[:10]

(1198,
 ['CEFACLOR',
  'CLOCORTOLONE PIVALATE',
  'SULFISOXAZOLE',
  'TIZANIDINE HYDROCHLORIDE',
  'BENAZEPRIL HYDROCHLORIDE',
  'OLANZAPINE',
  'TIGECYCLINE',
  'PROCAINAMIDE HYDROCHLORIDE',
  'ONDANSETRON HYDROCHLORIDE',
  'CHLOROPROCAINE HYDROCHLORIDE'])

In [101]:
list(difference)[:10]

['PYRILAMINE MALEATE',
 'ERYTHROMYCIN STEARATE',
 'DEXTROAMPHETAMINE ADIPATE',
 'DIETHYLSTILBESTROL',
 'DYCLONINE HYDROCHLORIDE',
 'TETRAHYDROZOLINE HYDROCHLORIDE',
 'KANAMYCIN SULFATE',
 'GENTIAN VIOLET',
 'TRIPROLIDINE HYDROCHLORIDE',
 'CEFUROXIME']

In [107]:
df_orig_andas[df_orig_andas['active_ingredients_split']=="DIETHYLSTILBESTROL"]

,application_number,active_ingredients,brand_name,year,sponsor_name,submission_class_code_description,submission_status,indications_first_sent,active_ingredients_split
4772,ANDA083003,DIETHYLSTILBESTROL,STILBESTROL,1973,TABLICAPS,None,AP,NaN,DIETHYLSTILBESTROL


In [106]:
df_orig_ndas[df_orig_ndas['active_ingredients_split'].str.contains("DIETHYLSTILBESTROL")]

,application_number,active_ingredients,brand_name,year,sponsor_name,submission_class_code_description,submission_status,indications_first_sent,active_ingredients_split


In [113]:
df_to_process = pd.concat([df_orig_ndas, df_only_in_andas], axis=0, ignore_index=True).drop_duplicates()
df_to_process.shape

(7030, 9)

In [114]:
df_to_process.to_csv("/scratch/sdonev/Preclinical_Pipeline/10_use_case_disease_focus/out/FDA_nda_anda_AP.csv",index=False)

### embed the drug terms

In [112]:
import sys
sys.path.append("../04_normalization")   # adjust path to your real folder
from neural_based_nen import main

In [115]:
data_dir =  "/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/04_normalization/data/"

In [ ]:
mapping_type = "drug"
col_to_map = "active_ingredients"
input_file = "/scratch/sdonev/Preclinical_Pipeline/10_use_case_disease_focus/out/FDA_nda_anda_AP.csv"
output_file = "/scratch/sdonev/Preclinical_Pipeline/10_use_case_disease_focus/out/FDA_nda_anda_AP_mapped_active_ingredients.csv"

main(mapping_type, col_to_map, data_dir, input_file, output_file, stats_dir=None)

Input file: /scratch/sdonev/Preclinical_Pipeline/10_use_case_disease_focus/out/FDA_nda_anda_AP.csv


/home/sdonev/.local/lib/python3.11/site-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Using terminology: umls
Using distance threshold: 8.2
Starting normalization for: DRUG with cdist 8.2
Loading embeddings from /shares/animalwelfare.crs.uzh/Preclinical_Pipeline/04_normalization/data/umls/embeddings with prefix UMLS_emb...
Loading term-id pairs from /shares/animalwelfare.crs.uzh/Preclinical_Pipeline/04_normalization/data/umls/umls_term_id_pairs_combined.json...
Found COMBINED embeddings file, loading that one...
